# Avaliação do modelo original

Primeiro registrar o ambiente, restaurar os conjuntos verificados e definir o protocolo. A comparação deve manter o mesmo modelo-base, instruções e critérios antes e depois do ajuste. O teste fica reservado para o final.

Este notebook está em construção por etapas. A inspeção do ambiente foi executada no Colab; os valores observados estão em `reports/colab_environment.json`. Nenhum modelo foi carregado nesta primeira célula.


In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("\nPacotes instalados:")
for pacote in ("torch", "transformers", "peft", "trl", "accelerate", "bitsandbytes", "datasets"):
    try:
        print(f"  {pacote}: {version(pacote)}")
    except PackageNotFoundError:
        print(f"  {pacote}: não instalado")
try:
    import torch
    print("\nGPU disponível:", torch.cuda.is_available())
    print("Versão CUDA do PyTorch:", torch.version.cuda)
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_properties(0)
        print("GPU:", gpu.name)
        print(f"Memória total: {gpu.total_memory / 1024**3:.1f} GiB")
    else:
        print("Confira se o ambiente está configurado com GPU.")
except ImportError:
    print("\nPyTorch não pôde ser importado.")


## Restauração dos conjuntos

Conferir os hashes conhecidos e carregar apenas treino e validação; teste reservado.


In [ ]:
import io
import json
import hashlib
import zipfile
from pathlib import Path
from google.colab import files

PASTA = Path("/content/triunfo_data")
hashes_esperados = {
    "train": "0a139d3a1b4eb5f0d2c7f854f6e5940570769e19e053f6950290e4ee2314c003",
    "validation": "cef828a8d7ca10e0988cc16d056c571e10f13caf6c87458074bae10c2766b954",
    "test": "bf4fb9eb1316dfc621eb1b1c1085fbb44d1750b017e5eb65104240bf420f356f",
}
necessarios = ["manifest.json"] + [f"{nome}.jsonl" for nome in hashes_esperados]
if not all((PASTA / nome).is_file() for nome in necessarios):
    print("Selecione o ZIP de backup do Triunfo.")
    enviados = files.upload()
    assert len(enviados) == 1, "Selecione somente o ZIP."
    conteudo_zip = next(iter(enviados.values()))
    with zipfile.ZipFile(io.BytesIO(conteudo_zip)) as arquivo_zip:
        assert set(arquivo_zip.namelist()) == set(necessarios)
        conteudos = {nome: arquivo_zip.read(nome) for nome in necessarios}
    for nome, esperado in hashes_esperados.items():
        observado = hashlib.sha256(conteudos[f"{nome}.jsonl"]).hexdigest()
        assert observado == esperado, f"Hash diferente em {nome}."
    PASTA.mkdir(parents=True, exist_ok=True)
    for nome, conteudo in conteudos.items():
        (PASTA / nome).write_bytes(conteudo)
    print("Backup restaurado.")
else:
    print("Arquivos encontrados; verificando integridade.")
manifesto_dados = json.loads((PASTA / "manifest.json").read_text(encoding="utf-8"))
assert manifesto_dados["dataset_revision"] == "6bda777c88d21e5a204703c1ee45597a8fa4f734"
assert manifesto_dados["seed"] == 42
for nome, esperado in hashes_esperados.items():
    observado = hashlib.sha256((PASTA / f"{nome}.jsonl").read_bytes()).hexdigest()
    assert observado == esperado, f"Hash diferente em {nome}."
    assert observado == manifesto_dados["files"][nome]["sha256"]
    print(f"{nome}: hash conferido")
def carregar_jsonl(caminho):
    with caminho.open(encoding="utf-8") as arquivo:
        return [json.loads(linha) for linha in arquivo if linha.strip()]
dados_treino = carregar_jsonl(PASTA / "train.jsonl")
dados_validacao = carregar_jsonl(PASTA / "validation.jsonl")
assert len(dados_treino) == 4469
assert len(dados_validacao) == 559
print("Treino carregado:", len(dados_treino))
print("Validação carregada:", len(dados_validacao))
print("Teste preservado no arquivo para a avaliação final.")


## Perfil de tokens no treino

Revisão fixada após a execução inicial. Medição somente no treino, preservando os gabaritos e sem truncamento. Os pesos do modelo não são carregados nesta célula.


In [ ]:
import json
import math
from pathlib import Path
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False, token=False,
)
INSTRUCAO_BASE = 'You are a tool-use assistant. Use only the available tools. When calling a tool, return exactly one call inside square brackets: [tool_name(argument=value)]. Copy the tool name exactly, including spaces or dots. Use Python literal syntax for argument values. If no available tool can fulfill the request, explain why. If required information is missing, ask for it. Do not invent missing information or tool results.'

def montar_mensagens(exemplo, incluir_resposta=False):
    ferramentas = json.dumps(exemplo["ferramentas"], ensure_ascii=False, sort_keys=True)
    mensagens = [
        {"role": "system", "content": INSTRUCAO_BASE + "\n\nAvailable tools:\n" + ferramentas},
        {"role": "user", "content": exemplo["pedido"]},
    ]
    if incluir_resposta:
        mensagens.append({"role": "assistant", "content": exemplo["resposta"]})
    return mensagens

tokens_treino = []
tokens_respostas = []
for numero, exemplo in enumerate(dados_treino, start=1):
    texto = tokenizer.apply_chat_template(
        montar_mensagens(exemplo, incluir_resposta=True),
        tokenize=False, add_generation_prompt=False,
    )
    tokens_treino.append(len(tokenizer(texto, add_special_tokens=False, truncation=False)["input_ids"]))
    tokens_respostas.append(len(tokenizer(exemplo["resposta"], add_special_tokens=False, truncation=False)["input_ids"]))
    if numero % 1000 == 0:
        print(f"Medidos {numero}/{len(dados_treino)} exemplos")
def percentil(valores, proporcao):
    ordenados = sorted(valores)
    indice = max(0, math.ceil(proporcao * len(ordenados)) - 1)
    return ordenados[indice]
perfil_tokens = {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "training_examples_measured": len(tokens_treino),
    "full_conversation_tokens": {
        "minimum": min(tokens_treino), "p50": percentil(tokens_treino, 0.50),
        "p90": percentil(tokens_treino, 0.90), "p95": percentil(tokens_treino, 0.95),
        "maximum": max(tokens_treino),
    },
    "reference_response_tokens": {"p95": percentil(tokens_respostas, 0.95), "maximum": max(tokens_respostas)},
    "examples_above_context_size": {
        str(limite): sum(tamanho > limite for tamanho in tokens_treino)
        for limite in (1024, 1536, 2048)
    },
    "system_instruction": INSTRUCAO_BASE,
}
pasta_execucao = Path("/content/triunfo_run")
pasta_execucao.mkdir(exist_ok=True)
(pasta_execucao / "tokenizer_profile.json").write_text(
    json.dumps(perfil_tokens, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
print(json.dumps(perfil_tokens, ensure_ascii=False, indent=2))
